# LookOutCV — Sample Notebook

This notebook demonstrates the two core logging classes:

| Class | Purpose |
|---|---|
| `DetectionLogger` | Log object-detection predictions (class, confidence, bounding box) alongside optional image-quality metrics |
| `ClassificationLogger` | Log image classification predictions (class, confidence) alongside optional image-quality metrics |
| `DataCollectionLogger` | Log standalone image-quality metrics without prediction data |

All data is persisted as Parquet files under `lookout_cv_logs/`.

## 0  Setup

Run the cell below to import dependencies and **clean any log files from previous runs** 


In [2]:
import shutil
import numpy as np
import pandas as pd

from look_out_cv import CVMetrics, DataCollectionLogger, DetectionLogger, ClassificationLogger

# Remove any logs from a previous runs
shutil.rmtree("lookout_cv_logs", ignore_errors=True)


## 1  Detection Logger

`DetectionLogger` stores one row per bounding-box prediction.  
Optional **image-quality metrics** (`CONTRAST`, `BLUR`, `ORIENTATION`, etc.) are computed from the raw image and appended automatically.

> **`buffer_size`** controls how many rows are batched before a disk write.  
> The default `1` is convenient for development; set it to `50–100` in production for much better throughput.

In [3]:
detection_logger = DetectionLogger(
    "OD_model",
    enabled_metrics=[CVMetrics.CONTRAST, CVMetrics.BLUR, CVMetrics.ORIENTATION],
    buffer_size=5
)

Log 5 synthetic predictions. 

In [4]:
sample_predictions = [
    {
        "image_name": "sample_001.jpg",
        "pred_class": "person",
        "confidence": 0.96,
        "bbox_x1": 42, "bbox_y1": 18, "bbox_x2": 188, "bbox_y2": 310,
        # Portrait image filled with a uniform mid-grey → contrast ≈ 0
        "image": np.full((320, 240, 3), 128, dtype=np.uint8),
    },
    {
        "image_name": "sample_002.jpg",
        "pred_class": "car",
        "confidence": 0.88,
        "bbox_x1": 75, "bbox_y1": 96, "bbox_x2": 420, "bbox_y2": 275,
        # Landscape image with random noise → high contrast & blur score
        "image": np.random.default_rng(7).integers(0, 256, (360, 640, 3), dtype=np.uint8),
    },
    {
        "image_name": "sample_003.jpg",
        "pred_class": "bicycle",
        "confidence": 0.79,
        "bbox_x1": 110, "bbox_y1": 54, "bbox_x2": 286, "bbox_y2": 330,
        "image": np.full((360, 480, 3), 200, dtype=np.uint8),
    },
    {
        "image_name": "tiny_object.jpg",
        "pred_class": "bottle_cap",
        "confidence": 0.91,
        "bbox_x1": 30, "bbox_y1": 28, "bbox_x2": 34, "bbox_y2": 33,
        # Square image (64×64) → orientation = 0.5
        "image": np.full((64, 64, 3), 150, dtype=np.uint8),
    },
    {
        "image_name": "dark_scene.jpg",
        "pred_class": "person",
        "confidence": 0.73,
        "bbox_x1": 80, "bbox_y1": 45, "bbox_x2": 145, "bbox_y2": 210,
        # Very dark landscape → low contrast, orientation = 1.0
        "image": np.full((240, 320, 3), 8, dtype=np.uint8),
    },
]

for prediction in sample_predictions:
    detection_logger.log_prediction(**prediction)

# Flush the in-memory buffer to disk before reading back.
detection_logger.flush()

stored_predictions = pd.read_parquet(detection_logger.parquet_file)
print(f"Parquet file : {detection_logger.parquet_file}")
print(f"Rows stored  : {len(stored_predictions)}")
print(f"Columns      : {stored_predictions.columns.tolist()}")
display(stored_predictions)

Parquet file : lookout_cv_logs\OD_model\OD_model_logs.parquet
Rows stored  : 5
Columns      : ['image_name', 'pred_class', 'confidence', 'bbox_x1', 'bbox_y1', 'bbox_x2', 'bbox_y2', 'contrast', 'blur', 'orientation']


,image_name,pred_class,confidence,bbox_x1,bbox_y1,bbox_x2,bbox_y2,contrast,blur,orientation
0,sample_001.jpg,person,0.96,42.0,18.0,188.0,310.0,0.000000,0.000000,0.0
1,sample_002.jpg,car,0.88,75.0,96.0,420.0,275.0,73.917343,48962.171875,1.0
2,sample_003.jpg,bicycle,0.79,110.0,54.0,286.0,330.0,0.000000,0.000000,1.0
3,tiny_object.jpg,bottle_cap,0.91,30.0,28.0,34.0,33.0,0.000000,0.000000,0.5
4,dark_scene.jpg,person,0.73,80.0,45.0,145.0,210.0,0.000000,0.000000,1.0


### 1.1  Query the logged data

Because the log is plain Parquet you can query it with any pandas expression.  
Here we extract only **high-confidence detections** (≥ 0.85), sorted by confidence.

In [5]:
high_confidence = stored_predictions.loc[
    stored_predictions["confidence"] >= 0.85,
    ["image_name", "pred_class", "confidence", "bbox_x1", "bbox_y1", "bbox_x2", "bbox_y2"],
].sort_values("confidence", ascending=False)

print(f"{len(high_confidence)} high-confidence detections (≥ 0.85):")
display(high_confidence)

3 high-confidence detections (≥ 0.85):


,image_name,pred_class,confidence,bbox_x1,bbox_y1,bbox_x2,bbox_y2
0,sample_001.jpg,person,0.96,42.0,18.0,188.0,310.0
3,tiny_object.jpg,bottle_cap,0.91,30.0,28.0,34.0,33.0
1,sample_002.jpg,car,0.88,75.0,96.0,420.0,275.0


## 2  Data Collection Logger

`DataCollectionLogger` is for logging **image quality metrics without prediction data** — useful for monitoring input data drift in a validation or pre-processing pipeline.

The metrics computed are the same `CVMetrics` enum values used by `DetectionLogger`.

In [6]:
data_logger = DataCollectionLogger(
    "image_metrics",
    enabled_metrics=[CVMetrics.CONTRAST, CVMetrics.BLUR, CVMetrics.ORIENTATION],
)

sample_images = {
    "image_001.jpg": np.full((320, 240, 3), 128, dtype=np.uint8),
    "image_002.jpg": np.random.default_rng(21).integers(0, 256, (360, 640, 3), dtype=np.uint8),
    "image_003.jpg": np.full((480, 360, 3), 200, dtype=np.uint8),
    "tiny_object_scene.jpg": np.full((64, 64, 3), 150, dtype=np.uint8),
    "dark_scene.jpg": np.full((240, 320, 3), 8, dtype=np.uint8),
}

for image_name, image in sample_images.items():
    data_logger.log_image(image_name, image)

# Flush before reading back.
data_logger.flush()

stored_image_metrics = pd.read_parquet(data_logger.parquet_file)
print(f"Saved {len(stored_image_metrics)} image metric rows to {data_logger.parquet_file}")
print(f"Columns: {stored_image_metrics.columns.tolist()}")
display(stored_image_metrics)

Saved 5 image metric rows to lookout_cv_logs\image_metrics\image_metrics_metrics.parquet
Columns: ['image_name', 'contrast', 'blur', 'orientation']


,image_name,contrast,blur,orientation
0,image_001.jpg,0.000000,0.000000,0.0
1,image_002.jpg,73.938538,48837.535156,1.0
2,image_003.jpg,0.000000,0.000000,0.0
3,tiny_object_scene.jpg,0.000000,0.000000,0.5
4,dark_scene.jpg,0.000000,0.000000,1.0


### 2.1  Query the image metrics

Find images with **very low contrast** (e.g. uniform or near-black frames) that may indicate a camera or pre-processing problem.

In [7]:
low_contrast = stored_image_metrics.loc[
    stored_image_metrics["contrast"] < 10,
    ["image_name", "contrast", "blur", "orientation"],
].sort_values("contrast")

print(f"{len(low_contrast)} low-contrast images (contrast < 10):")
display(low_contrast)

4 low-contrast images (contrast < 10):


,image_name,contrast,blur,orientation
0,image_001.jpg,0.0,0.0,0.0
2,image_003.jpg,0.0,0.0,0.0
3,tiny_object_scene.jpg,0.0,0.0,0.5
4,dark_scene.jpg,0.0,0.0,1.0


## 3  Context Manager Usage

Both loggers support the `with` statement.  
The buffer is **automatically flushed** when the block exits.

In [8]:
# Flush is guaranteed even if an exception is raised mid-loop.
with DetectionLogger("ctx_model", enabled_metrics=[CVMetrics.BRIGHTNESS]) as logger:
    logger.log_prediction(
        image_name="img_a.jpg",
        pred_class="dog",
        confidence=0.92,
        bbox_x1=10, bbox_y1=10, bbox_x2=200, bbox_y2=300,
        image=np.full((300, 200, 3), 180, dtype=np.uint8),
    )


import os, pyarrow.parquet as pq

ctx_file = os.path.join("lookout_cv_logs", "ctx_model", "ctx_model_logs.parquet")
print(pq.read_table(ctx_file).to_pandas().to_string(index=False))

image_name pred_class  confidence  bbox_x1  bbox_y1  bbox_x2  bbox_y2  brightness
 img_a.jpg        dog        0.92     10.0     10.0    200.0    300.0       180.0


## 4  Classification Logger

`ClassificationLogger` stores one row per classification prediction.
Optional image-quality metrics are computed from the raw image and appended automatically.

In [9]:
from look_out_cv import ClassificationLogger

classification_logger = ClassificationLogger(
    "Cls_model",
    enabled_metrics=[CVMetrics.CONTRAST, CVMetrics.BLUR],
    buffer_size=3
)

Log 3 synthetic classification predictions.

In [10]:
classification_samples = [
    {
        "image_name": "cls_001.jpg",
        "pred_class": "cat",
        "confidence": 0.95,
        "image": np.random.default_rng(1).integers(0, 256, (224, 224, 3), dtype=np.uint8),
    },
    {
        "image_name": "cls_002.jpg",
        "pred_class": "dog",
        "confidence": 0.88,
        "image": np.full((224, 224, 3), 100, dtype=np.uint8),
    },
    {
        "image_name": "cls_003.jpg",
        "pred_class": "bird",
        "confidence": 0.99,
        "image": np.random.default_rng(2).integers(0, 256, (224, 224, 3), dtype=np.uint8),
    }
]

for sample in classification_samples:
    classification_logger.log_prediction(**sample)

# Flush
classification_logger.flush()

stored_cls_predictions = pd.read_parquet(classification_logger.parquet_file)
print(f"Parquet file : {classification_logger.parquet_file}")
print(f"Rows stored  : {len(stored_cls_predictions)}")
print(f"Columns      : {stored_cls_predictions.columns.tolist()}")
display(stored_cls_predictions)

Parquet file : lookout_cv_logs\Cls_model\Cls_model_logs.parquet
Rows stored  : 3
Columns      : ['image_name', 'pred_class', 'confidence', 'contrast', 'blur']


,image_name,pred_class,confidence,contrast,blur
0,cls_001.jpg,cat,0.95,74.028305,48567.550781
1,cls_002.jpg,dog,0.88,0.000000,0.000000
2,cls_003.jpg,bird,0.99,73.949158,49508.800781
